# Arkenstone Discovery V8 — Stage A: ARK-017 mechanism factorial

Dissects **why** protection works. ARK-015 gave a reliable failure generator: after order-invariant
binding was acquired, canonical-only continuation at HIGH LR collapsed SEALED order robustness in
8/8 runs while LOW and fully-augmented HIGH held at 0/8. ARK-017 separates the two candidate causes:

- **update magnitude** (HIGH LR capped to LOW's applied parameter movement), vs
- **invariant-supporting replay** (HIGH + 1/16 order-diverse data), vs both together.

Six arms, 3 fresh acquisition seeds, 8k-step continuation, SEALED robust-retention endpoint.
Expected T4 budget ~180 minutes. Run cells top to bottom on a **T4 GPU**.
Do not modify the pinned commit or thresholds after seeing results.

In [ ]:
import os, shutil, subprocess, sys, torch
PINNED_RUNNER_COMMIT = '59e1b805b7d93b7f2e1e9d3ea66b34c4fabca9c8'
REPO = '/content/An-Ra-the-new-AGI-discovery-v8a'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','50','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
paths = [
 'experiments/COLAB/discovery_v7_common.py',
 'experiments/ARK-017/run_ark017.py',
 'experiments/ARK-015/run_ark015.py',
 'experiments/ARK-011/run_ark011.py',
 'experiments/ARK-014/run_ark014.py',
 'experiments/ARK-001/run_ark001.py',
 'experiments/lib/ark_tasks.py',
]
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
print('compile gate: PASS on', len(paths), 'files')
runner = os.path.join(REPO,'experiments/ARK-017/run_ark017.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
subprocess.run([sys.executable, runner, '--smoke-test', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env, check=True)
print('SMOKE TEST: PASS — proceed to the next cell.')

In [ ]:
# Full campaign. 180 minutes is a safety budget, not a minimum runtime.
import os, subprocess, sys
runner = os.path.join(REPO,'experiments/ARK-017/run_ark017.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--budget-minutes', '180', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env)
print('FULL CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('A failure receipt and all completed partial JSONs are in the result ZIP. Do not rerun before inspecting them.')

In [ ]:
from pathlib import Path
root = Path('/content/arkenstone_ark017_results')
print('Result files:')
for p in sorted(root.glob('*')):
    print(p.name, p.stat().st_size if p.is_file() else '')
verdict = root / 'ARK-017_RESULT.json'
if verdict.exists():
    print()
    print('ARK-017 RESULT:')
    print(verdict.read_text())
zip_path = root / 'ARKENSTONE_ARK017_RESULTS.zip'
if zip_path.exists():
    print()
    print('ZIP:', zip_path, zip_path.stat().st_size, 'bytes')
    print('Download and share this ZIP for receipt validation.')
else:
    print('ZIP not found - inspect the JSON receipts above.')